# Modelling a glacier with IGM — tutorial (Session 2)

This tutorial carries out a series of **numerical experiments with IGM** on the **[Rhone Glacier](https://en.wikipedia.org/wiki/Rhone_Glacier)** (Valais, ~15 km², the source of the Rhone river; RGI ID `RGI2000-v7.0-G-11-01706`, set by default in all parameter files). Step by step, you calibrate the model on observations, reconstruct the glacier since the Little Ice Age, project it until 2100, and explore two key concepts (the SMB model, the response time).

It is the **basis of your mini-project**: once you have gone through it with the Rhone Glacier, do the same with **your own glacier** (put its RGI ID in each file of `experiment/`, and keep the same steps).

**Follow this notebook from top to bottom.** It lives in **your own copy** of the template, `~/work/my-glacier/` (copied once from `~/work/glamod/session2/my-glacier/`). It only **reads and plots** the results of your IGM runs: re-run it as often as you like.

| What | Where |
|---|---|
| **Set** the parameters | YAML files in `experiment/` (text editor) |
| **Run** IGM | a **Terminal**, from this folder: `cd ~/work/my-glacier` |
| **Look** at the results | this notebook, reading `outputs/<run>/` |

Steps:
1. **Present**: download the data, check the initial state
2. **Calibration of the ice-flow parameters** (`arrhenius`, `tau_ref`) on the observed speeds
3. **Past**: Little Ice Age → today, **calibration of the SMB** (`melt_enhancer`)
4. **Future**: climate scenarios 2020–2100
5. **A simple SMB model** (ELA-based)
6. **Response time**

Session 3 (moraines, figures for the report): `tutorial-session3.ipynb`.

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/params_past.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here

## 1. Present: download the data and check the initial state

**What happens here**: `oggm_shop` downloads everything IGM needs for the glacier: the topography, the glacier outline, an estimate of the ice thickness (Millan et al., 2022), the observed surface speeds and the climate (1901–2019). Before any experiment, we check that these data make sense (thickness, speeds, climate trend), since the whole project relies on them.

▶ **In the terminal**: a 1-year run downloads the data (≈ 1 min the first time):

```bash
igm_run +experiment=params_present
```

In [ ]:
plot_input_data(INPUT)
plot_climate(RGI_ID)

## 2. Calibration of the ice-flow parameters

**Why?** The ice flow depends on two poorly known parameters: the ice softness (`arrhenius`, deformation) and the basal sliding (`tau_ref`). The modelled speeds must match the **observed** ones, otherwise the glacier moves too much or too little ice downstream, and its evolution is wrong. We calibrate these two parameters **once, on today's glacier**, and then keep them for all other experiments.

First, compare the speeds modelled with the default parameters (`arrhenius` = 78, `tau_ref` = 0.215) with the observed ones (Millan et al., 2022):

In [ ]:
compare_modeled_observed_velocity(INPUT, "outputs/params_present/output.nc", year=2020)

Then explore **all combinations** of ice softness (`arrhenius`) and basal sliding (`tau_ref`) with 1-year runs. `-m` (multirun) makes one run per combination of the values given in the command line (3 × 4 = 12 runs, ≈ 3 min).

▶ **In the terminal**:

```bash
igm_run -m +experiment=params_sensitivity processes.iceflow.physics.viscosity.arrhenius=25,50,100 processes.iceflow.physics.sliding.tau_ref=0.1,0.2,0.4,0.8
```

Results: `outputs/params_sensitivity/0`, `1`, …, `11`. Each map below shows the speed difference (modelled − observed) of one run: look for a **bias close to 0** and a small RMSE (in red). Refine the values around the best one if needed (re-run with other values: the folder is overwritten).

In [ ]:
runs, labels = sweep_runs("outputs/params_sensitivity")   # the 12 runs
compare_velocity_misfit(INPUT, runs, year=2020)

> ✅ **Calibration 1 — ice flow** (double-click to edit): `arrhenius` = ...  `tau_ref` = ...
>
> (example, Rhone Glacier: `arrhenius` = 30, `tau_ref` = 0.315)
>
> ➡️ Put these values in the `iceflow → physics` block of **all** the files of `experiment/`.

## 3. Past: from the Little Ice Age to today — calibration of the SMB

**Why?** The SMB computed from the climate (temperature, precipitation) is uncertain. We calibrate it by checking that the model reproduces the **known history** of the glacier: an advance until the Little Ice Age maximum (~1850, marked by moraines), then the retreat until today. If the model gets the past right, we can trust it more for the future.

**Experiment**: we do not know the glacier geometry in 1700, so we start from **today's** geometry with a cooler pre-industrial climate: the glacier grows to its LIA maximum, then retreats with the observed climate 1901–2019. A single parameter is calibrated: `melt_enhancer` (more or less melt).

Start in 1700 from today's geometry, advance to the LIA maximum (~1850), retreat with the observed climate 1901–2019 (≈ a few minutes). Keep the pre-1901 climate (`clim_trend_array`) of `experiment/params_past.yaml` and **adjust `melt_enhancer`** (start: 0.8, explore 0.5–2) until the glacier (i) reaches the LIA moraines around 1850 (Siegfried map on map.geo.admin.ch) and (ii) matches today's ice volume in 2020.

▶ **In the terminal**:

```bash
igm_run +experiment=params_past
```

To keep several tries, copy the file (e.g. `params_past_v2.yaml`, and change its run folder at the end of the file to `outputs/params_past_v2`) and add the run to `past_runs` below.

In [ ]:
run = "outputs/params_past"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

In [ ]:
# Ice thickness through time, and along the main flowline
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")
plot_flowline_evolution(os.path.join(run, "output.nc"), rgi_id=RGI_ID)

In [ ]:
# Compare the volume of several past runs with today's volume (ice thickness of the input data)
ds = xr.open_dataset(INPUT); dx = float(ds.x[1] - ds.x[0])
obs = {"vol": ([2020], [float(ds.thk.sum()) * dx**2 / 1e9])}
past_runs = ["outputs/params_past"]              # <- add your other past runs here
compare_runs(past_runs, [os.path.basename(r) for r in past_runs], observations=obs)

> ✅ **Calibration 2 — SMB** (double-click to edit): `melt_enhancer` = ...
>
> ➡️ Put this value in `params_future.yaml` (and `params_past_moraines.yaml` for Session 3).

## 4. Future: climate scenarios 2020–2100

**Why?** With the calibrated ice flow (section 2) and SMB (section 3), we project the glacier until 2100 under several **climate scenarios** (from "committed", i.e. today's climate kept constant, to RCP8.5, strong warming). The comparison tells how much of the glacier is already lost whatever we do, and how much depends on future emissions.

`delta_temp` is relative to **1960–1990** (see Lecture 2):
- 2020 row (in `experiment/params_future.yaml`): present-day offset (2000–2019, printed by `plot_climate` above)
- 2100: 1981–2010 offset + CH2018 warming of the scenario, given **in the command line**, one value per scenario (committed = today's offset, RCP2.6, RCP4.5, RCP8.5)

▶ Check that `params_future.yaml` contains your calibrations 1 and 2, then **in the terminal** (4 runs, a few minutes; values here for the Rhone Glacier, adapt them):

```bash
igm_run -m +experiment=params_future processes.climate.oggm.clim_trend_array.2.1=1.3,2.0,3.4,5.1
```

One run per scenario: `outputs/params_future/0`, `1`, `2`, `3` (in the order of the list).

In [ ]:
runs, labels = sweep_runs("outputs/params_future")    # the scenario runs, labelled by their 2100 delta_temp
compare_runs(runs, labels)

In [ ]:
# Maps for one scenario (here the third one: RCP4.5)
animate_glacier_evolution(runs[2] + "/output.nc", variable="thk")

## 5. A simple SMB model: ELA-based

**Why?** The climate-driven SMB hides many processes (temperature, precipitation, melt factors). Here we take a simpler approach: we prescribe the **ELA directly**. This forces us to think about **which ELAs must have prevailed** to drive the glacier evolution we observe (LIA advance, then retreat).

**Goal**: try to reproduce the volume curve of your calibrated past run (section 3) by playing with the ELAs (and gradients) in `experiment/params_past_simple.yaml` (today's ELA: end-of-summer snowline on map.geo.admin.ch, GLAMOS; LIA ELAs ≈ 100–200 m lower).

**Difficulty**: a glacier does not react immediately. It has a **response time**: after an ELA change, it takes years to decades to adjust its geometry. So the ELA must change *before* the volume does. This is studied in section 6.

▶ **In the terminal**:

```bash
igm_run +experiment=params_past_simple
```

In [ ]:
plot_smb_simple(ela=3000, gradabl=0.009, gradacc=0.005, accmax=2.0)   # <- your ELA and gradients
compare_runs(["outputs/params_past", "outputs/params_past_simple"], ["climate-driven SMB", "ELA-based SMB"], observations=obs)

## 6. Response time (your own IGM module, oscillating ELA)

Here the ELA oscillates like a sine wave (period 200 yr, amplitude 200 m) and we look at how the glacier volume follows.

**Look at the module first**: this SMB is not part of IGM, it is a **user module** written in Python: `user/code/processes/smb_oscillating.py` (its parameters: `user/conf/processes/smb_oscillating.yaml`). Open it: like any IGM module, it has 3 functions, `initialize` (called once), `update` (at each time step: computes the ELA at time t, then the SMB from the surface elevation) and `finalize`. This is how you can extend IGM with your own processes.

▶ **In the terminal** (400 model years, a few minutes). For another period, copy `experiment/params_response_time.yaml` to `experiment/params_response_100.yaml`, set `period: 100` and the run folder `outputs/params_response_100` in it:

```bash
igm_run +experiment=params_response_time
igm_run +experiment=params_response_100
```

In [ ]:
plot_response_time("outputs/params_response_time")
# plot_response_time("outputs/params_response_100")

**What you should see**:
- The volume **lags behind** the ELA: the volume is maximal some decades **after** the ELA is lowest (e.g. ~45 years for the Rhone Glacier, period 200 yr).
- With a **shorter** period (50–100 yr), the glacier has no time to adjust: the volume oscillation is **small** and strongly delayed. With a **longer** period (400 yr), the glacier nearly follows the ELA: **large** oscillation.
- The lag gives an estimate of the **response time** of your glacier. Compare with $\tau \approx H / (-b_{\text{terminus}})$ (Lecture 1a; $H$: typical thickness, $b_{\text{terminus}}$: SMB at the front) and with the 1D tutorial of Session 1.

> ✅ **Conclusion — response time** (double-click to edit): the response time of my glacier is about ... years. Consequence for the future: by 2100, the glacier is still / no longer responding to the warming of today because ...